# Popolazione agent_features — Feature di rete (NetworkX)

Questo notebook calcola le feature che richiedono il grafo conversazionale:
- `in_degree`, `out_degree`
- `pagerank`
- `betweenness`
- `local_clustering`
- `egonet_size`, `egonet_density`
- `reciprocity_local`

Le feature SQL-based sono già state popolate da `feature.py`.

## 1. Caricamento archi e costruzione grafo

In [ ]:
import sqlite3
import pandas as pd
import networkx as nx

conn = sqlite3.connect("../data/moltbook.db")
conn.row_factory = sqlite3.Row

# Carica gli archi con una sola query — il JOIN sfrutta idx_comments_parent_id
edges = pd.read_sql("""
    SELECT c.author_name AS source, p.author_name AS target
    FROM comments c
    JOIN comments p ON c.parent_id = p.id
    WHERE c.parent_id IS NOT NULL
      AND c.author_name IS NOT NULL
      AND p.author_name IS NOT NULL
""", conn)

print(f"Archi caricati: {len(edges)}")
print(f"Autori unici (source): {edges['source'].nunique()}")
print(f"Autori unici (target): {edges['target'].nunique()}")

In [ ]:
# Costruzione del grafo diretto
G = nx.from_pandas_edgelist(edges, source="source", target="target", create_using=nx.DiGraph())

print(f"Nodi: {G.number_of_nodes()}")
print(f"Archi: {G.number_of_edges()}")

## 2. Calcolo feature veloci: degree e pagerank

In [ ]:
# Degree
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())

print(f"In-degree calcolato per {len(in_deg)} nodi")
print(f"Out-degree calcolato per {len(out_deg)} nodi")

In [ ]:
# PageRank
pagerank = nx.pagerank(G)

print(f"PageRank calcolato per {len(pagerank)} nodi")
print(f"Top 5 per PageRank:")
for name, pr in sorted(pagerank.items(), key=lambda x: -x[1])[:5]:
    print(f"  {name}: {pr:.6f}")

## 3. UPDATE nel database

In [ ]:
# Mappa agent_name -> agent_id per l'UPDATE
agent_ids = pd.read_sql("SELECT id, name FROM agents", conn)
name_to_id = dict(zip(agent_ids["name"], agent_ids["id"]))

print(f"Agenti nel DB: {len(name_to_id)}")
print(f"Agenti nel grafo: {G.number_of_nodes()}")

In [ ]:
# UPDATE: degree + pagerank
cursor = conn.cursor()
updated = 0

for name in G.nodes():
    agent_id = name_to_id.get(name)
    if not agent_id:
        continue
    cursor.execute("""
        UPDATE agent_features
        SET in_degree = ?, out_degree = ?, pagerank = ?
        WHERE agent_id = ?
    """, (in_deg[name], out_deg[name], pagerank[name], agent_id))
    updated += 1

conn.commit()
print(f"Aggiornati {updated} agenti con degree + pagerank")

## 4. Betweenness (lenta — esecuzione esatta)

In [ ]:
print("Calcolo betweenness centrality (potrebbe richiedere tempo)...")
betweenness = nx.betweenness_centrality(G, normalized=True)
print(f"Betweenness calcolata per {len(betweenness)} nodi")

# UPDATE nel DB
cursor = conn.cursor()
updated = 0
for name, val in betweenness.items():
    agent_id = name_to_id.get(name)
    if not agent_id:
        continue
    cursor.execute("UPDATE agent_features SET betweenness = ? WHERE agent_id = ?", (val, agent_id))
    updated += 1

conn.commit()
print(f"Aggiornati {updated} agenti con betweenness")

## 5. Local clustering

In [ ]:
clustering = nx.clustering(G)
print(f"Local clustering calcolato per {len(clustering)} nodi")

cursor = conn.cursor()
updated = 0
for name, val in clustering.items():
    agent_id = name_to_id.get(name)
    if not agent_id:
        continue
    cursor.execute("UPDATE agent_features SET local_clustering = ? WHERE agent_id = ?", (val, agent_id))
    updated += 1

conn.commit()
print(f"Aggiornati {updated} agenti con local_clustering")

## 6. Egonet (size + density) e reciprocity locale

In [ ]:
cursor = conn.cursor()
updated = 0

for name in G.nodes():
    agent_id = name_to_id.get(name)
    if not agent_id:
        continue

    # Egonet: sottografo dei vicini diretti + il nodo stesso
    ego = nx.ego_graph(G, name)
    egonet_size = ego.number_of_nodes() - 1  # escludi il nodo stesso
    egonet_density = nx.density(ego) if egonet_size > 0 else 0.0

    # Reciprocity locale: frazione di archi uscenti che sono reciprocati
    successors = set(G.successors(name))
    predecessors = set(G.predecessors(name))
    reciprocated = len(successors & predecessors)
    reciprocity_local = reciprocated / len(successors) if successors else 0.0

    cursor.execute("""
        UPDATE agent_features
        SET egonet_size = ?, egonet_density = ?, reciprocity_local = ?
        WHERE agent_id = ?
    """, (egonet_size, egonet_density, reciprocity_local, agent_id))
    updated += 1

conn.commit()
print(f"Aggiornati {updated} agenti con egonet + reciprocity")

## 7. Verifica finale

In [ ]:
# Controlla quanti agenti hanno le feature di rete popolate
check = pd.read_sql("""
    SELECT
        COUNT(*) as totale,
        COUNT(in_degree) as con_in_degree,
        COUNT(pagerank) as con_pagerank,
        COUNT(betweenness) as con_betweenness,
        COUNT(local_clustering) as con_clustering,
        COUNT(egonet_size) as con_egonet,
        COUNT(reciprocity_local) as con_reciprocity
    FROM agent_features
""", conn)

print(check.T.to_string(header=False))

In [ ]:
conn.close()
print("Connessione chiusa.")